# Aufgabe 3 – Rekurrente Modelle und WaveNet vergleichen

In dieser Aufgabe vergleichen Sie GRU, LSTM und ein WaveNet-ähnliches Modell auf
derselben Wettervorhersage. Im Mittelpunkt steht die Frage, wie rekurrente
Zustände und kausale, dilatierte Faltungen zeitliche Abhängigkeiten erfassen.

Die Architekturdetails sind nicht vorgegeben. Sie müssen jedoch Kausalität,
Receptive Field und faire Trainingsbedingungen nachvollziehbar prüfen.


## 1. Gemeinsame Datenbasis

Die Daten werden direkt geladen und stündlich abgetastet. Verwenden Sie für alle
drei Modelle dieselben Merkmale, dieselbe chronologische Aufteilung und dieselben
Fenster. Als gemeinsame Ausgangslage dienen 64 Eingabestunden und ein
Prognosehorizont von sechs Stunden.

Bereiten Sie die Daten ohne Leakage vor und bestimmen Sie eine naive Baseline.
Dokumentieren Sie Merkmale, Formen und die Fehlerkennzahlen der Baseline. Wenn Sie
von der vorgeschlagenen Merkmalsauswahl abweichen, begründen Sie dies.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

DATA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"

weather_raw = pd.read_csv(DATA_URL, compression="zip")
weather_raw["Date Time"] = pd.to_datetime(
    weather_raw["Date Time"], format="%d.%m.%Y %H:%M:%S"
)
weather_hourly = (
    weather_raw.iloc[5::6]
    .set_index("Date Time")
    .tail(40_000)
    .copy()
)

wind_columns = ["wv (m/s)", "max. wv (m/s)"]
weather_hourly[wind_columns] = (
    weather_hourly[wind_columns]
    .replace(-9999.0, np.nan)
    .interpolate(limit_direction="both")
)

def make_windows(values, target_index, width, horizon, stride=3):
    starts = np.arange(0, len(values) - width - horizon + 1, stride)
    X = np.stack([values[i:i + width] for i in starts])
    y = values[starts + width + horizon - 1, target_index, None]
    return X, y

def temperature_mae(y_true, y_pred, temperature_std):
    return float(np.mean(np.abs(y_true - y_pred)) * temperature_std)


## 2. Drei passende Architekturen entwickeln

Entwerfen Sie jeweils ein Modell für:

1. **GRU:** eine rekurrente Architektur, die das Fenster zu einem Zielwert
   zusammenfasst,
2. **LSTM:** eine vergleichbare, aber eigenständig begründete Architektur,
3. **WaveNet-ähnliches Modell:** ein Stapel eindimensionaler kausaler Faltungen
   mit wachsenden Dilationsraten.

Für das WaveNet-Modell sind `Conv1D`, kausales Padding und Dilationsraten
verpflichtend. Filterzahl, Kernelgröße, Zahl und Reihenfolge der Dilationsraten,
Residual-Verbindungen, Dropout und Ausgabekopf wählen Sie selbst.

Versuchen Sie, den Vergleich interpretierbar zu halten. Sie können entweder
ähnliche Parameterzahlen anstreben oder bewusst unterschiedlich große Modelle
vergleichen. Erklären Sie in beiden Fällen, welche Aussage der Vergleich erlaubt.
Notieren Sie vor der Implementierung die erwartete Ein- und Ausgabeform jeder
Schichtengruppe.


## 3. Kausalität und Receptive Field prüfen

Für einen seriellen Stapel kausaler Faltungen mit Kernelgröße $k$, Stride 1 und
Dilationsraten $d_1,\dots,d_L$ gilt ohne zusätzliche Sprünge:

$$R = 1 + (k - 1)\sum_{l=1}^{L} d_l.$$

Berechnen Sie das Receptive Field Ihrer gewählten WaveNet-Architektur. Falls Sie
mehrere Blöcke, andere Kernelgrößen oder eine abweichende Struktur verwenden,
leiten Sie die Reichweite passend zu Ihrem Modell her.

Beantworten Sie:

1. Kann die letzte Modellausgabe theoretisch von allen 64 Eingabestunden
   beeinflusst werden?
2. Was bedeutet es, wenn das Receptive Field kleiner als das Eingabefenster ist?
3. Warum verhindert `padding="causal"` den Zugriff auf spätere Positionen?
4. Weshalb wäre `padding="same"` nicht automatisch kausal?
5. Vergrößert ein größeres Receptive Field zwangsläufig die
   Vorhersagequalität?
6. Wie könnten Sie experimentell prüfen, ob weit zurückliegende Stunden wirklich
   genutzt werden?


## 4. Vergleichsexperiment planen

Legen Sie gemeinsame Trainingsregeln fest: Optimizer, Loss, maximale Epochen,
Batchgröße, Early Stopping, Seed und Auswahlkennzahl. Kleine
architekturspezifische Anpassungen sind erlaubt, wenn sie begründet und
dokumentiert werden.

Trainieren Sie zunächst je ein Hauptmodell. Führen Sie danach mindestens eine
gezielte Zusatzuntersuchung durch, zum Beispiel:

- kleineres gegen größeres Receptive Field,
- mit gegen ohne Residual-Verbindungen,
- flaches gegen tieferes rekurrentes Modell,
- ähnliche Parameterzahl für alle Architekturen,
- Einfluss einer veränderten Fensterlänge.

Verwenden Sie Validierungsdaten für Entscheidungen und den Testabschnitt erst für
den finalen Vergleich.


## 5. Modelle auswerten

Berichten Sie für Baseline, GRU, LSTM und WaveNet mindestens:

- Validierungs- und Test-MAE sowie RMSE in Grad Celsius,
- trainierbare Parameter,
- Epochen bis zum besten Validierungsergebnis,
- Trainingsdauer oder eine vergleichbare Aufwandsgröße.

Wählen Sie bei Bedarf eigene Visualisierungen. Entscheiden Sie für jede
Darstellung vorher, welche Frage sie beantworten soll. Ein einzelner
Sammelplot ist nicht zwingend sinnvoll, wenn Fehler, Parameterzahl und Zeit sehr
unterschiedliche Skalen besitzen.


## 6. Architekturvergleich und Reflexion

Beantworten Sie anhand Ihrer Ergebnisse:

1. Welches Modell war am genauesten, welches am effizientesten?
2. Wie transportieren GRU und LSTM Information über die Zeit?
3. Wie erfasst das WaveNet-Modell frühere Werte ohne rekurrenten Hidden State?
4. Welche Rolle spielt der zusätzliche Zellzustand der LSTM?
5. Welche Architektur lässt sich über Zeitpositionen grundsätzlich besser
   parallelisieren?
6. Hat das Modell mit den meisten Parametern gewonnen?
7. Welche Beobachtung aus Ihrer Zusatzuntersuchung war überraschend?
8. Welche Vorteile oder Nachteile zeigten Residual-Verbindungen beziehungsweise
   größere Dilationsraten?
9. Welche Aussage ist durch Ihre Messwerte belegt, und welche wäre nur eine
   allgemeine Vermutung über GRU, LSTM oder WaveNet?
10. Wie würden Sie das Experiment ändern, wenn die Vorhersage 24 statt sechs
    Stunden in die Zukunft reichen soll?

Schließen Sie mit einer kurzen Modellentscheidung ab. Nennen Sie dabei nicht nur
den Fehler, sondern auch Modellgröße, Laufzeit und Grenzen des Experiments.
